# Workforce Analytics – Klassifikation und Schwellenwerte

Diese Demo greift Methoden aus einem Machine-Learning-Studienprojekt auf:
explorative Datenanalyse, ein Modellvergleich und die Wahl eines Schwellenwerts
anhand unterschiedlich gewichteter Fehler. Alle Zeilen werden aus einem festen
Zufalls-Seed erzeugt. Die Modellgüte und Kosten beziehen sich auf diese Simulation.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    PrecisionRecallDisplay, average_precision_score, confusion_matrix, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

rng = np.random.default_rng(42)
n = 4000
data = pd.DataFrame({
    "employment_years": rng.uniform(0, 20, n).round(1),
    "workload": rng.integers(1, 6, n),
    "schedule": rng.choice(["fixed", "flex", "shift"], n),
})
logit = (
    -2.8 + 0.47 * data["workload"] - 0.09 * data["employment_years"]
    + 0.55 * data["schedule"].eq("shift")
    + 0.7 * data["workload"].eq(5) * data["schedule"].eq("shift")
)
data["simulated_exit"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
data.groupby("workload")["simulated_exit"].agg(["size", "mean"])

## Training, Validierung und Test

Das Training lernt die Modellparameter. Die Validierung bestimmt Modell und
Schwellenwert. Der Testsatz bleibt bis zur Auswertung zurückgehalten.
Ein übersehener positiver Fall kostet in dieser Simulation vier Einheiten,
ein Fehlalarm eine Einheit.

In [ ]:
X = data.drop(columns="simulated_exit")
y = data["simulated_exit"]
X_rest, X_test, y_rest, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_rest, y_rest, test_size=0.25, stratify=y_rest, random_state=42,
)
numeric = ["employment_years", "workload"]
categorical = ["schedule"]
models = {
    "Logistische Regression": LogisticRegression(),
    "Entscheidungsbaum": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=80, max_leaf_nodes=12, random_state=42,
    ),
}
candidates = []

def counts_at_threshold(labels, probabilities, threshold):
    decisions = probabilities >= threshold
    _, fp, fn, _ = confusion_matrix(labels, decisions, labels=[0, 1]).ravel()
    return int(fp), int(fn)

for name, estimator in models.items():
    prep = ColumnTransformer([
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
    ])
    model = make_pipeline(prep, estimator).fit(X_train, y_train)
    scores = model.predict_proba(X_val)[:, 1]
    for threshold in np.linspace(0, 1, 101):
        fp, fn = counts_at_threshold(y_val, scores, threshold)
        candidates.append((fp + 4 * fn, name, float(threshold), model))

cost, name, threshold, model = min(candidates, key=lambda row: row[:3])
pd.DataFrame(
    [(label, min(row[0] for row in candidates if row[1] == label)) for label in models],
    columns=["Modell", "Niedrigste Validierungskosten"],
)

In [ ]:
test_scores = model.predict_proba(X_test)[:, 1]
rows = []
for label, cutoff in [("Standard", 0.5), ("Auf Validierung gewählt", threshold)]:
    fp, fn = counts_at_threshold(y_test, test_scores, cutoff)
    rows.append((label, cutoff, fp, fn, fp + 4 * fn))
print(f"{name} | ROC-AUC {roc_auc_score(y_test, test_scores):.3f} "
      f"| Average Precision {average_precision_score(y_test, test_scores):.3f}")
display(pd.DataFrame(rows, columns=[
    "Entscheidungsregel", "Schwellenwert", "Fehlalarme", "Übersehene Fälle", "Kosten",
]))
PrecisionRecallDisplay.from_predictions(y_test, test_scores)
plt.show()

## Einordnung

Der Kostensatz ist eine Annahme. Die Simulation zeigt, wie er die
Entscheidungsregel beeinflusst und wie sich das auf bisher ungesehenen Zeilen
messen lässt. Die erzeugten Zusammenhänge beschreiben keine realen Beschäftigten.
Die Demo dient der Modellbewertung und eignet sich ausschließlich zu Lernzwecken.